# 🎓 Agente de Reembolsos

Lee el listado de solicitudes de reintegro y, para cada una:

| `estado` en la base | Acción |
|---|---|
| `cobro_duplicado` | Ejecuta la devolución directamente |
| `pago_correcto` | Informa que **no** corresponde la devolución |
| cualquier otro | Solicita supervisión humana (el grafo se **pausa**) |

**Regla de diseño:** el ruteo lo decide el campo `estado` de la base, no el texto del LLM.
El LLM redacta la justificación que queda en el log de auditoría; nunca decide el camino.


## Paso 1: Instalación de Ollama

In [ ]:
# Instalamos Ollama (el motor que corre el modelo localmente)
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q langgraph langchain-core requests
print("Dependencias instaladas.")

## Paso 2: Levantar el servidor de Ollama y descargar el modelo

Ollama funciona como un servidor: lo prendemos en segundo plano (`subprocess.Popen`) y después le pedimos que descargue el modelo `phi3:mini`. Esto puede tardar 1-2 minutos la primera vez.

In [ ]:
import subprocess, time, requests

# Apagamos cualquier instancia previa por las dudas, y prendemos el servidor en segundo plano
subprocess.run(["pkill", "ollama"], stderr=subprocess.DEVNULL)
time.sleep(2)
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)
time.sleep(5)

# Descargamos el modelo (liviano, ideal para este ejercicio)
!ollama pull phi3:mini

print("Ollama listo con el modelo phi3:mini.")

## Paso 3: Función para hablar con Ollama

El LLM es opcional para el funcionamiento del agente: si falla, el grafo igual decide bien,
porque la decisión no depende de él. Por eso la función captura los errores en vez de propagarlos.

In [ ]:
import requests

def consultar_ollama(prompt: str, model: str = "phi3:mini") -> str:
    """Envía un prompt a Ollama y devuelve el texto de la respuesta.

    Si Ollama no está disponible devuelve un texto de fallback en vez de romper:
    el agente no puede quedar bloqueado por el LLM, que acá solo redacta prosa.
    """
    url = "http://localhost:11434/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False, "options": {"temperature": 0.1}}
    try:
        respuesta = requests.post(url, json=payload, timeout=120)
        respuesta.raise_for_status()
        return respuesta.json()["response"].strip()
    except Exception as e:
        return f"(LLM no disponible: {type(e).__name__}: {e})"

# Prueba rápida
print(consultar_ollama("Respondé en una sola palabra: ¿2+2 es par o impar?"))

## Paso 4: Cargar los datos (CSV)

Levanta `facturas.csv` de Colab. Si no lo tenés subido al entorno, se genera uno de respaldo.

In [ ]:
import pandas as pd
import os

CSV_PATH = "/content/facturas.csv"

if not os.path.exists(CSV_PATH):
    print("⚠️  No encontré facturas.csv. Genero uno de respaldo.")
    filas = [
        {"invoice_id": "F-2024-901", "cliente": "Laura Fernández", "monto": 57267.83, "estado": "cobro_excesivo"},
        {"invoice_id": "F-2024-902", "cliente": "Martín Gómez",    "monto": 2557.99,  "estado": "cobro_duplicado"},
        {"invoice_id": "F-2024-903", "cliente": "Sofía Rodríguez", "monto": 9078.56,  "estado": "cobro_duplicado"},
        {"invoice_id": "F-2024-904", "cliente": "Diego Pérez",     "monto": 853.69,   "estado": "pago_rechazado"},
        {"invoice_id": "F-2024-905", "cliente": "Valentina López", "monto": 3476.33,  "estado": "pago_correcto"},
    ]
    pd.DataFrame(filas).to_csv(CSV_PATH, index=False)

df_facturas = pd.read_csv(CSV_PATH)
print(f"Cargadas {len(df_facturas)} facturas.")
print(df_facturas["estado"].value_counts())
display(df_facturas.head())

In [ ]:
def herramienta_sql_facturacion(invoice_id: str) -> dict:
    """Simula la API SQL: busca la factura en el CSV cargado.

    Normaliza el estado (strip + lower) para que la clasificación no dependa
    de mayúsculas ni de espacios accidentales en la base.
    """
    fila = df_facturas[df_facturas["invoice_id"] == invoice_id]
    if fila.empty:
        return {"monto": 0.0, "estado": "no_encontrada"}
    registro = fila.iloc[0]
    return {
        "monto": float(registro["monto"]),
        "estado": str(registro["estado"]).strip().lower(),
    }

print("Herramienta 'sql_facturacion' lista.")
print(herramienta_sql_facturacion("F-2024-902"))
print(herramienta_sql_facturacion("F-0000-000"))  # caso no encontrado

## Paso 5: Clasificación de la solicitud

Acá está el corazón de la corrección. La decisión es **determinística** y sale del campo
`estado` de la base. Las constantes son las mismas claves que después usa el mapa de
`add_conditional_edges`, así no hay forma de que la función de ruteo devuelva algo que el
grafo no sepa interpretar.

In [ ]:
# Claves de ruteo: se usan TANTO en la función de decisión COMO en el mapa del grafo.
DEVOLVER    = "devolver"
NO_CORRESPONDE = "no_corresponde"
SUPERVISION = "supervision"

def clasificar_estado(estado: str) -> str:
    """Traduce el estado de la base a una de las tres decisiones posibles."""
    e = str(estado).strip().lower()
    if e == "cobro_duplicado":
        return DEVOLVER
    if e == "pago_correcto":
        return NO_CORRESPONDE
    return SUPERVISION          # 'el resto de los casos': default explícito

# Chequeo rápido de la tabla de decisión
for e in ["cobro_duplicado", "COBRO_DUPLICADO ", "pago_correcto", "cobro_excesivo",
          "pago_rechazado", "no_encontrada", ""]:
    print(f"{e!r:20} -> {clasificar_estado(e)}")

## Paso 6: Creación del sandbox

Como Colab no puede correr `dockerd`, se emula ejecutando el script directamente con `subprocess`.
El sandbox **no decide** si corresponde la devolución: eso ya se decidió en el Paso 5.
Solo valida que el monto sea ejecutable de forma automática.

In [ ]:
import os

os.makedirs("sandbox_docker", exist_ok=True)

# 1. El script que hace el cálculo — esto es lo que correría DENTRO del contenedor
script = '''import sys, json

LIMITE_AUTOMATICO = 50000

def probar_reembolso(monto):
    if monto <= 0:
        return {"ok": False, "razon": "monto invalido"}
    if monto > LIMITE_AUTOMATICO:
        return {"ok": False, "razon": "excede el limite permitido"}
    return {"ok": True, "monto_validado": monto}

if __name__ == "__main__":
    monto = float(sys.argv[1])
    resultado = probar_reembolso(monto)
    print(json.dumps(resultado))
'''
with open("sandbox_docker/calculo_reembolso.py", "w") as f:
    f.write(script)

# 2. El Dockerfile real — define cómo se empaqueta el sandbox en un contenedor aislado
dockerfile = '''FROM python:3.11-slim
WORKDIR /app
COPY calculo_reembolso.py .
# --network=none al correrlo evita que el contenedor tenga acceso a internet: aislamiento real
ENTRYPOINT ["python", "calculo_reembolso.py"]
'''
with open("sandbox_docker/Dockerfile", "w") as f:
    f.write(dockerfile)

print("Archivos generados en sandbox_docker/:")
print(os.listdir("sandbox_docker"))

In [ ]:
import subprocess, json

def nodo_sandbox(monto: float) -> dict:
    """
    En producción, esto sería: docker run --rm --network=none sandbox-image {monto}
    En Colab (sin dockerd), corremos el mismo script como subproceso aislado.
    """
    print(f"  🧪 [Sandbox] Probando reembolso de ${monto}...")
    proceso = subprocess.run(
        ["python", "sandbox_docker/calculo_reembolso.py", str(monto)],
        capture_output=True, text=True
    )
    if proceso.returncode != 0 or not proceso.stdout.strip():
        return {"ok": False, "razon": f"sandbox fallo: {proceso.stderr.strip()[:200]}"}
    resultado = json.loads(proceso.stdout.strip())
    print(f"  Resultado: {resultado}")
    return resultado

print("Nodo sandbox (basado en el script Docker) definido.")

## Paso 7: Definición del grafo (LangGraph)

```
                      ┌─ cobro_duplicado ─→ sandbox ─┬─ ok ──→ reembolso_real ──→ END
leer_solicitud        │                              └─ no ──→ supervision_humana (PAUSA)
      ↓               │
 analisis_llm ─ router ┼─ pago_correcto ───→ informar_no_corresponde ──→ END
                      │
                      └─ resto ──────────→ supervision_humana (PAUSA)
```

Tres detalles que hacen que esto funcione:

1. El router devuelve las **mismas constantes** que son claves del mapa de `add_conditional_edges`.
2. `supervision_humana` es un **nodo real**, no un `END` disfrazado: queda registrado en la auditoría.
3. `interrupt_before=["supervision_humana"]` hace que el grafo **se pause de verdad** antes de ese nodo.

In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import datetime, uuid, operator

class AgentState(TypedDict):
    invoice_id: str
    monto_factura: float
    estado_factura: str
    decision: str            # DEVOLVER | NO_CORRESPONDE | SUPERVISION
    motivo: str
    sandbox_ok: bool
    analisis_llm: str
    # El reducer 'operator.add' concatena los eventos de cada nodo automáticamente:
    # ningún nodo necesita leer y reescribir el log completo.
    log_auditoria: Annotated[List[dict], operator.add]

def evento(fase: str, detalle: str) -> dict:
    return {
        "id": str(uuid.uuid4()),
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "fase": fase,
        "detalle": detalle,
    }

In [ ]:
def paso_leer_solicitud(state: AgentState) -> dict:
    """Lee la solicitud de la base y fija la decisión. Ninguna IA participa acá."""
    factura = herramienta_sql_facturacion(state["invoice_id"])
    decision = clasificar_estado(factura["estado"])
    print(f"📄 {state['invoice_id']}: estado='{factura['estado']}' monto=${factura['monto']} → decisión '{decision}'")
    return {
        "monto_factura": factura["monto"],
        "estado_factura": factura["estado"],
        "decision": decision,
        "motivo": "",
        "log_auditoria": [evento(
            "LECTURA_SOLICITUD",
            f"estado={factura['estado']}; monto={factura['monto']}; decision={decision}",
        )],
    }

def paso_analisis_llm(state: AgentState) -> dict:
    """El LLM redacta la justificación para la auditoría. NO decide el camino."""
    prompt = f"""Sos un analista de reembolsos. La decisión YA fue tomada por el sistema;
tu única tarea es redactar una justificación breve para el registro de auditoría.

- Factura: {state['invoice_id']}
- Estado en el sistema de pagos: {state['estado_factura']}
- Monto: ${state['monto_factura']}
- Decisión tomada: {state['decision']}

Escribí UNA sola frase explicando por qué esa decisión es correcta. Sin encabezados ni listas."""
    texto = consultar_ollama(prompt)
    print(f"🧠 Justificación del LLM: {texto}")
    return {
        "analisis_llm": texto,
        "log_auditoria": [evento("JUSTIFICACION_LLM", texto)],
    }

def paso_sandbox(state: AgentState) -> dict:
    resultado = nodo_sandbox(state["monto_factura"])
    return {
        "sandbox_ok": bool(resultado["ok"]),
        "motivo": resultado.get("razon", ""),
        "log_auditoria": [evento("ACCION_SANDBOX_DOCKER", str(resultado))],
    }

def paso_reembolso_real(state: AgentState) -> dict:
    print(f"  💸 [API real] Ejecutando devolución de ${state['monto_factura']}...")
    return {"log_auditoria": [evento(
        "DEVOLUCION_EJECUTADA",
        f"Devolución de ${state['monto_factura']} ejecutada automáticamente "
        f"(estado={state['estado_factura']}, sandbox ok).",
    )]}

def paso_informar_no_corresponde(state: AgentState) -> dict:
    print(f"  ℹ️  NO corresponde devolución para {state['invoice_id']}: el pago es correcto.")
    return {"log_auditoria": [evento(
        "INFORME_NO_CORRESPONDE",
        f"No corresponde devolución: estado={state['estado_factura']}.",
    )]}

def paso_supervision_humana(state: AgentState) -> dict:
    print(f"  👤 Supervisión humana registrada para {state['invoice_id']}.")
    return {"log_auditoria": [evento(
        "SUPERVISION_HUMANA",
        f"Caso derivado a un humano: estado={state['estado_factura']}; "
        f"motivo={state['motivo'] or 'estado no automatizable'}.",
    )]}

print("Nodos definidos.")

In [ ]:
def ruta_por_decision(state: AgentState) -> str:
    """Router principal. Devuelve una de las tres constantes de ruteo — nunca otra cosa."""
    return state["decision"]

def ruta_post_sandbox(state: AgentState) -> str:
    """Si el sandbox rechaza el monto, el caso va a supervisión en vez de perderse."""
    if state["sandbox_ok"]:
        return "reembolso_real"
    print(f"  ❌ Sandbox rechazó ({state['motivo']}) → supervisión humana.")
    return SUPERVISION

workflow = StateGraph(AgentState)
workflow.add_node("leer_solicitud", paso_leer_solicitud)
workflow.add_node("analisis_llm", paso_analisis_llm)
workflow.add_node("sandbox", paso_sandbox)
workflow.add_node("reembolso_real", paso_reembolso_real)
workflow.add_node("informar_no_corresponde", paso_informar_no_corresponde)
workflow.add_node("supervision_humana", paso_supervision_humana)

workflow.add_edge(START, "leer_solicitud")
workflow.add_edge("leer_solicitud", "analisis_llm")

workflow.add_conditional_edges(
    "analisis_llm",
    ruta_por_decision,
    {
        DEVOLVER: "sandbox",
        NO_CORRESPONDE: "informar_no_corresponde",
        SUPERVISION: "supervision_humana",
    },
)
workflow.add_conditional_edges(
    "sandbox",
    ruta_post_sandbox,
    {
        "reembolso_real": "reembolso_real",
        SUPERVISION: "supervision_humana",
    },
)

workflow.add_edge("reembolso_real", END)
workflow.add_edge("informar_no_corresponde", END)
workflow.add_edge("supervision_humana", END)

memoria_grafo = MemorySaver()
# interrupt_before: el grafo se detiene ANTES de supervision_humana y espera a la persona.
app = workflow.compile(checkpointer=memoria_grafo, interrupt_before=["supervision_humana"])

print("Grafo compilado. ✅")

In [ ]:
# Visualización del grafo. Degrada a texto si no hay red o faltan dependencias de dibujo.
grafo = app.get_graph()
try:
    from IPython.display import Image, display
    display(Image(grafo.draw_mermaid_png()))
except Exception:
    try:
        print(grafo.draw_ascii())
    except Exception:
        print(grafo.draw_mermaid())

## Paso 8: Procesar TODO el listado de solicitudes

Una solicitud = un `thread_id` propio. Reusar el mismo `thread_id` entre facturas
mezcla estados de casos distintos en el checkpointer.

In [ ]:
resultados = []

print("====== EJECUCIÓN DEL AGENTE SOBRE TODO EL LISTADO ======\n")
for invoice_id in df_facturas["invoice_id"]:
    print(f"--- {invoice_id} " + "-" * 40)
    config = {"configurable": {"thread_id": f"caso-{invoice_id}"}}
    app.invoke({"invoice_id": invoice_id, "log_auditoria": []}, config)

    snapshot = app.get_state(config)
    pendiente = snapshot.next  # () si terminó; ('supervision_humana',) si está pausado
    if pendiente:
        print(f"  🔒 PAUSADO esperando a un humano. Próximo paso: {pendiente}")
    resultados.append({
        "invoice_id": invoice_id,
        "estado": snapshot.values["estado_factura"],
        "decision": snapshot.values["decision"],
        "pausado": bool(pendiente),
        "fases": " → ".join(e["fase"] for e in snapshot.values["log_auditoria"]),
    })
    print()

display(pd.DataFrame(resultados))

## Paso 9: Aprobación humana de los casos pausados

Los casos derivados a supervisión quedaron esperando. Acá una persona los revisa y
el grafo retoma desde donde quedó, dejando constancia en el log.

In [ ]:
pendientes = [r["invoice_id"] for r in resultados if r["pausado"]]
print(f"👤 Casos esperando supervisión humana: {pendientes or 'ninguno'}\n")

for invoice_id in pendientes:
    config = {"configurable": {"thread_id": f"caso-{invoice_id}"}}
    print(f"--- {invoice_id}: un humano revisó el caso ---")
    app.invoke(None, config)          # retoma desde el punto de interrupción
    print(f"  Estado final: {app.get_state(config).next or 'finalizado'}\n")

print("✅ Flujo completado.")

## Paso 10: Log de auditoría

Un registro por solicitud, con la decisión, el camino recorrido y la justificación del LLM.

In [ ]:
import json

auditoria = []
for invoice_id in df_facturas["invoice_id"]:
    config = {"configurable": {"thread_id": f"caso-{invoice_id}"}}
    estado_final = app.get_state(config).values
    auditoria.append({
        "caso_id": estado_final["invoice_id"],
        "estado_origen": estado_final["estado_factura"],
        "decision": estado_final["decision"],
        "traza_completa": estado_final["log_auditoria"],
    })

log_para_auditoria = {
    "sistema": "agente_reembolsos_ollama_docker_v2",
    "referencia_norma": "ISO 42001 - Trazabilidad de decisiones automatizadas",
    "casos_procesados": len(auditoria),
    "casos": auditoria,
}

print(json.dumps(log_para_auditoria, indent=2, ensure_ascii=False))

## Paso 11: Verificación de la tabla de decisión

Comprueba que cada caso terminó donde la consigna dice que tiene que terminar.

In [ ]:
ESPERADO = {
    "cobro_duplicado": "DEVOLUCION_EJECUTADA / SUPERVISION_HUMANA (si el sandbox rechaza)",
    "pago_correcto":   "INFORME_NO_CORRESPONDE",
}

errores = 0
for caso in auditoria:
    fases = [e["fase"] for e in caso["traza_completa"]]
    estado = caso["estado_origen"]

    if estado == "cobro_duplicado":
        ok = "DEVOLUCION_EJECUTADA" in fases or "SUPERVISION_HUMANA" in fases
    elif estado == "pago_correcto":
        ok = "INFORME_NO_CORRESPONDE" in fases
    else:
        ok = "SUPERVISION_HUMANA" in fases

    marca = "✅" if ok else "❌"
    errores += 0 if ok else 1
    print(f"{marca} {caso['caso_id']:12} estado={estado:16} fases={' → '.join(fases[2:])}")

print(f"\n{'TODO OK' if errores == 0 else str(errores) + ' CASO(S) MAL RUTEADO(S)'}")